<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [11]</a>'.</span>

# 10 — draft persistence: an edit a person made is never lost

The safety half of `docs/architecture/editing-performance.md`, and the half
that matters. Every check here reads the value back rather than a flag: a
`saveState` of `saved` is not evidence, the string in `localStorage` or the
string the agent answers is.

**Two checks are expected to FAIL in this run** and both are acceptance of a
change that has not been made yet:

- the persisted payload still carries the media catalogue (change 2 of the
  plan takes it out);
- three field writes inside one quiet window still become three commits
  instead of one (change 3 makes them batch).

Everything else must be green **today**, because everything else is the CMS's
promise as it already stands. If one of those goes red, the change has broken
something and no latency number makes up for it.

## How each scenario is driven, and why

A scenario that can be driven in node is driven in node, because a node check
runs anywhere: no browser, no build, no credentials, no writes on a live site.
`support/editing-harness.ts` bundles `packages/jaen/src/redux` itself, so the
store, the persister, the recorder and the flusher are the real ones; only the
browser globals and the agent's HTTP are replaced, and the replaced `fetch` is
what makes "offline" mean what it means to the client, a rejected call rather
than an error answer.

One scenario per process. The store is a module singleton, so a second
scenario in the same process would inherit the first one's state. A reload is
two processes over one storage file, which is exactly what a reload is.

What cannot be driven in node is driven in a browser against the live agent:
a real reload, a real hidden tab, a real offline context and a second editor.
Those run on a local production build of booklimo.at signed in as the booklimo
human admin, they make commits on booklimo through the live agent, and they set
the field back and read it back out of an emptied browser at the end. Nothing
is written on limosen.

In [1]:
import json, os, pathlib

import jaen_testkit as k

k.start_run('10-draft-persistence')

REPO = pathlib.Path(k.CONFIG['repo_root'])
SITE = pathlib.Path(os.environ.get('JAEN_SITE_BUILD', '/home/snekmin/git/limosen-v3/booklimo.at'))
LIVE = SITE / 'jaen-data' / 'live.json'
MEDIA = SITE / 'jaen-data' / 'live-media.json'
WORK = pathlib.Path(os.environ.get('JAEN_WORK_DIR', '/tmp/jaen-editing'))
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE = WORK / 'editing-harness.cjs'
PLAYWRIGHT_PYTHON = os.environ.get(
    'JAEN_PLAYWRIGHT_PYTHON', '/home/snekmin/git/taxi-app/tests/.venv/bin/python')

with k.section('the harness'):
    with k.check('the harness bundles out of the jaen source') as c:
        esbuild = REPO / 'node_modules' / '.bin' / 'esbuild'
        if not esbuild.is_file():
            c.skip('no esbuild in the checkout')
        r = c.require(k.sh(
            '%s tests/support/editing-harness.ts --bundle --platform=node '
            '--format=cjs --target=node20 --outfile=%s --log-level=warning'
            % (esbuild, BUNDLE), cwd=str(REPO), timeout=180))
        c.expect_true(BUNDLE.is_file(), 'bundled %d B' % BUNDLE.stat().st_size)


def harness(scenario, env=None, timeout=120):
    # One scenario, one process.
    base = {'JAEN_HARNESS_LIVE': str(LIVE), 'JAEN_HARNESS_MEDIA': str(MEDIA)}
    base.update(env or {})
    return k.sh('node %s %s' % (BUNDLE, scenario), cwd=str(REPO), env=base,
                timeout=timeout, label='harness %s' % scenario)


def scenario(name, env=None, timeout=120):
    r = harness(name, env, timeout)
    return (json.loads(r.text) if r.ok and r.text.startswith('{') else None), r

## What the persisted payload carries

The plan wants three things of it: the outbox, so a change that has not
reached the agent survives a reload; the local edits, for the same reason; and
**not** the media catalogue, which is 99% of it, is never edited by hand and
comes back on the next poll.

The scenario writes a field with the agent unreachable, so the change is still
in the outbox when the payload is read. The catalogue check is the acceptance
of change 2 and is red until that change is made.

In [2]:
PAYLOAD, _r = scenario('payload')

with k.section('the persisted payload'):
    with k.check('the payload carries the unsent change and the edit') as c:
        if PAYLOAD is None:
            c.fail('the harness did not answer', abort=True)
        c.expect_equal(PAYLOAD['localEdit'], 'the payload check',
                       'the edit is in the payload, read back out of it')
        c.expect_equal(PAYLOAD['outboxLength'], 1, 'changes waiting in the outbox')
        c.expect_equal(PAYLOAD['outboxKinds'], ['fieldWrite'], 'what is waiting')

    with k.check('the payload does not carry the media catalogue') as c:
        if PAYLOAD is None:
            c.skip('the harness did not answer')
        c.note('%d B payload, %d media nodes in it, %d B of catalogue'
               % (PAYLOAD['bytes'], PAYLOAD['mediaNodeCount'], PAYLOAD['catalogueBytes']))
        c.expect_equal(PAYLOAD['mediaNodeCount'], 0,
                       'change 2 of the plan: the catalogue is not persisted')

print(json.dumps(PAYLOAD, indent=1))

{
 "scenario": "payload",
 "bytes": 77390,
 "hasLocalEdit": true,
 "localEdit": "the payload check",
 "outboxLength": 1,
 "outboxKinds": [
  "fieldWrite"
 ],
 "mediaNodeCount": 140,
 "catalogueBytes": 76168,
 "keys": [
  "site",
  "page",
  "status",
  "popup",
  "widget",
  "remote"
 ]
}


## An edit followed by a reload

Two processes over one storage file. The first writes a field and ends; the
second builds the store from what is on disk, the way `loadState` does on a
fresh page, and is asked what the field says.

In [3]:
STORE_FILE = WORK / 'reload-store.json'
STORE_FILE.unlink(missing_ok=True)

with k.section('a reload'):
    with k.check('an edit survives the browser being started again') as c:
        first, r1 = scenario('editThenExit', {'JAEN_HARNESS_STORE': str(STORE_FILE),
                                              'JAEN_HARNESS_VALUE': 'survives a reload'})
        if first is None:
            c.fail('the first process did not answer: %s' % r1.evidence(), abort=True)
        second, r2 = scenario('readBack', {'JAEN_HARNESS_STORE': str(STORE_FILE)})
        if second is None:
            c.fail('the second process did not answer: %s' % r2.evidence(), abort=True)
        c.expect_equal(second['value'], 'survives a reload',
                       'the value the second process read out of storage')
        c.note('and %d media nodes came back with it' % second['mediaNodeCount'])

## An edit followed by the tab going away

The dangerous one. Today the write is synchronous inside the dispatch, so the
value is in storage in the same frame. After change 1 it will be written in an
idle callback with a 250 ms timeout, and `visibilitychange` to hidden and
`pagehide` will write at once. The check is written so that it holds for both:
the value must be in storage once the tab is hidden with no waiting at all, and
in any case within the idle deadline the plan gives itself.

In [4]:
HIDDEN, _r = scenario('hiddenTab')

with k.section('a hidden tab'):
    with k.check('a hidden tab has the edit in storage with no waiting') as c:
        if HIDDEN is None:
            c.fail('the harness did not answer', abort=True)
        c.note('%d listener(s) heard visibilitychange' % HIDDEN['listeners'])
        c.expect_equal(HIDDEN['afterHidden'], 'typed and then the tab went away',
                       'read back out of storage in the same frame as the event')

    with k.check('the edit is in storage within the idle deadline anyway') as c:
        if HIDDEN is None:
            c.skip('the harness did not answer')
        c.expect_equal(HIDDEN['afterIdleDeadline'], 'typed and then the tab went away',
                       'read back 300 ms after the dispatch')
        if HIDDEN['immediately'] is None:
            c.warn('the write is asynchronous: nothing was in storage in the '
                   'dispatch\'s own frame, which is what change 1 makes true')
        else:
            c.note('the write is synchronous today: it was there at once')

## An edit made offline

The outbox is the offline queue, and it is the offline queue because
`persist-state` writes it. The scenario writes a field with `fetch` rejecting,
reads the value and the queue back out of storage, then lets the agent come
back.

Two ways back are measured. Without any help the client retries on its own
backoff, and the first retry is measured here rather than assumed: the plan and
`draft-state.md` both say the first retry is at two seconds, and the code
indexes `RETRY_SECONDS` with a failure count it has already incremented, so it
is at five. That is written down as an observation, not fixed here.

In [5]:
OFFLINE, _r = scenario('offlineDrain', timeout=180)
ONLINE, _r2 = scenario('onlineEvent', timeout=120)

with k.section('offline'):
    with k.check('an edit made offline is kept, in the store and in storage') as c:
        if OFFLINE is None:
            c.fail('the harness did not answer', abort=True)
        while_offline = OFFLINE['whileOffline']
        c.expect_equal(while_offline['persistedValue'], 'written while offline',
                       'read back out of storage while the agent is unreachable')
        c.expect_equal(while_offline['outboxLength'], 1, 'waiting in the outbox')
        c.expect_equal(while_offline['saveState'], 'offline', 'what the toolbar says')

    with k.check('the queue drains on its own when the agent comes back') as c:
        if OFFLINE is None:
            c.skip('the harness did not answer')
        after = OFFLINE['afterReturn']
        c.expect_equal(after['outboxLength'], 0, 'the outbox is empty again')
        c.expect_equal(after['persistedValue'], 'written while offline',
                       'and the value is still the one that was typed')
        c.note('drained %.1f s after the network came back' % (OFFLINE['drainedAfterMs'] / 1000))

    with k.check('the documented first retry is the one the code takes') as c:
        if OFFLINE is None:
            c.skip('the harness did not answer')
        seconds = OFFLINE['drainedAfterMs'] / 1000
        if 1.5 <= seconds <= 3.5:
            c.ok('first retry at %.1f s, as documented' % seconds)
        else:
            c.warn('first retry at %.1f s, while draft-state.md and the plan both '
                   'say 2 s: RETRY_SECONDS is indexed with an already incremented '
                   'failure count. Nothing is lost by it, the queue only waits '
                   'longer than the document claims.' % seconds)

    with k.check('the browser saying online drains it at once') as c:
        if ONLINE is None:
            c.fail('the harness did not answer', abort=True)
        c.expect_equal(ONLINE['outboxLength'], 0, 'the outbox is empty')
        c.expect_equal(ONLINE['persistedValue'], 'written while offline',
                       'the value read back out of storage')
        c.note('drained %d ms after the online event' % ONLINE['drainedAfterMs'])

## A save inside the quiet window is one commit

Change 3 of the plan: the quiet window applies to every field write and not
only to a streaming one, so a person moving through three fields makes one
commit instead of three. The scenario writes three different fields 120 ms
apart, which is well inside the proposed 1,500 ms window, and counts the calls
the agent received.

Red today, on purpose: today only an MDX-shaped field waits, so this is three
commits.

In [6]:
QUIET, _r = scenario('quietWindow', timeout=180)

with k.section('the quiet window'):
    with k.check('three field writes inside the window are one commit') as c:
        if QUIET is None:
            c.fail('the harness did not answer', abort=True)
        c.note('%d call(s) to the agent, %s change(s) each'
               % (QUIET['saveCalls'], QUIET['changesPerCall']))
        c.expect_equal(QUIET['saveCalls'], 1, 'change 3 of the plan')

    with k.check('none of the three values is lost, whatever the batching') as c:
        if QUIET is None:
            c.skip('the harness did not answer')
        c.expect_equal(QUIET['values'],
                       {'FleetTitle': 'one', 'ServicesTitle': 'two', 'AboutTitle': 'three'},
                       'all three read back out of the store')
        c.expect_equal(QUIET['outboxLength'], 0, 'and all three left the outbox')

## A poll arriving mid-edit

The poller hydrates the remote draft into the same store the editor is typing
into. `hydrate` folds the outbox back on top of the remote answer, so a change
that has not reached the agent yet cannot be overwritten by an answer that does
not contain it. The scenario keeps the agent unreachable so the change stays
unsent, then hydrates the draft as the poller would and asks the store and the
storage what the field says.

In [7]:
POLL, _r = scenario('pollMidEdit')

with k.section('a poll mid-edit'):
    with k.check('a poll does not overwrite an unsent change') as c:
        if POLL is None:
            c.fail('the harness did not answer', abort=True)
        c.note('the remote draft says %r' % POLL['remoteValueBeforeMerge'])
        c.expect_equal(POLL['value'], 'typed and not yet sent',
                       'the store after the hydration')
        c.expect_equal(POLL['persistedValue'], 'typed and not yet sent',
                       'and storage, read back')
        c.expect_equal(POLL['outboxLength'], 1, 'the change is still waiting')

## Discard clears the outbox and nothing else

`RESET_STATE` keeps `active`, the head and the authors and drops everything a
browser had not published. With the agent configured the toolbar no longer
offers it, but the action is still there and the escape below relies on it.

In [8]:
DISCARD, _r = scenario('discard')

with k.section('discard'):
    with k.check('discard drops the unsent changes and keeps the head') as c:
        if DISCARD is None:
            c.fail('the harness did not answer', abort=True)
        c.expect_equal(DISCARD['before']['outboxLength'], 1, 'something to discard')
        c.expect_equal(DISCARD['after']['outboxLength'], 0, 'the outbox is empty')
        c.expect_equal(DISCARD['after'].get('value'), None,
                       'the edit is gone from the store, read back')
        c.expect_equal(DISCARD['after'].get('persistedValue'), None,
                       'and out of storage')
        c.expect_equal(DISCARD['after']['headSha'], 'harness-head-poll',
                       'the head the poller gave us survives')
        c.expect_equal(DISCARD['after']['authors'], 1, 'and so do the authors')

## The escape: no agent at all

The plan's rollback. A site built without the `agent` plugin option behaves
exactly as it did before the shared draft: no recorder, no flusher, nothing on
the network, and `localStorage` as the only store. The scenario runs the same
harness with the define removed.

In [9]:
NOAGENT, _r = scenario('noAgent', {'JAEN_HARNESS_AGENT': '0'})

with k.section('the escape'):
    with k.check('without the agent the CMS still saves to localStorage alone') as c:
        if NOAGENT is None:
            c.fail('the harness did not answer', abort=True)
        c.expect_true(not NOAGENT['agentConfigured'], 'no agent is configured')
        c.expect_equal(NOAGENT['persistedValue'], 'saved without the agent',
                       'the edit read back out of storage')
        c.expect_equal(NOAGENT['saveCalls'], 0, 'and nothing left the browser')
        c.expect_equal(NOAGENT['outboxLength'], 0, 'no outbox is kept')

## The four the browser has to answer

A real reload, a real hidden tab, a real offline context and a second editor,
on a local production build of booklimo.at against the live agent, signed in as
the booklimo human admin. The run makes commits on booklimo, sets the field
back to the value it found and proves it by loading the site again in a browser
whose storage was emptied, so the value it reports as read back is the
repository's answer.

It SKIPs cleanly when playwright, the build or the human admin's credentials
are missing, the way the rest of the suite skips.

In [10]:
SAFETY = None

with k.section('the browser'):
    with k.check('a reload, a hidden tab, an offline edit and a second editor') as c:
        if not os.path.isfile(PLAYWRIGHT_PYTHON):
            c.skip('no playwright interpreter at %s' % PLAYWRIGHT_PYTHON)
        if not (SITE / 'public' / 'index.html').is_file():
            c.skip('no production build of booklimo.at in %s' % (SITE / 'public'))
        if not os.path.isfile(os.path.expanduser('~/.config/taxi-app/humans.env')):
            c.skip('no booklimo human admin configured')
        r = c.require(k.sh('%s support/editing-browser.py safety \'{}\'' % PLAYWRIGHT_PYTHON,
                           cwd=str(REPO / 'tests'), timeout=1200, label='browser safety'),
                      'the browser')
        SAFETY = json.loads(r.text)
        if SAFETY.get('skipped'):
            c.skip(SAFETY['skipped'])
        c.expect_true(SAFETY.get('original') is not None, 'the field had a value to begin with')

    with k.check('an edit survives the tab being hidden and the page reloaded') as c:
        if not SAFETY or SAFETY.get('skipped'):
            c.skip('the browser half did not run')
        expected = (SAFETY['original'] or '') + ' hidden'
        c.expect_equal(SAFETY['afterHidden'], expected, 'in storage when the tab went away')
        c.expect_equal(SAFETY['afterReload'], expected, 'and still there after the reload')

    with k.check('an edit made offline is kept and drains when the network is back') as c:
        if not SAFETY or SAFETY.get('skipped'):
            c.skip('the browser half did not run')
        offline = SAFETY['offline']
        expected = (SAFETY['original'] or '') + ' offline'
        c.expect_equal(offline['value'], expected, 'read back while offline')
        c.expect_true(offline['outbox'] >= 1, '%d change(s) waiting' % offline['outbox'])
        c.expect_true(offline['drained'], 'the queue drained')
        c.expect_equal(offline['valueAfterDrain'], expected, 'and the value is unchanged')

    with k.check('a second editor reads the change out of the repository') as c:
        if not SAFETY or SAFETY.get('skipped'):
            c.skip('the browser half did not run')
        c.expect_equal(SAFETY['secondEditorSees'], (SAFETY['original'] or '') + ' offline',
                       'what the second browser hydrated from the agent')

    with k.check('the live field is back to the value the run found') as c:
        if not SAFETY or SAFETY.get('skipped'):
            c.skip('the browser half did not run')
        # The invariant on the live site: the browser's storage was emptied and
        # this value came from the agent.
        c.expect_equal(SAFETY['readBack'], SAFETY['original'],
                       'read back out of an emptied browser')

print(json.dumps(SAFETY, indent=1))

{
 "scenario": "safety",
 "original": "Our fleeting",
 "afterHidden": "Our fleeting hidden",
 "afterReload": "Our fleeting hidden",
 "offline": {
  "value": "Our fleeting offline",
  "outbox": 1,
  "saveState": "offline",
  "drained": true,
  "valueAfterDrain": "Our fleeting offline"
 },
 "secondEditorSees": "Our fleeting offline",
 "readBack": "Our fleeting"
}


## What this run leaves for the reviewer

- The two red checks are the plan's changes 2 and 3 and nothing else. Every
  other check is green against the code as it stands, which is the point: the
  CMS does not lose an edit today, and the change must not make that untrue.
- The offline retry observation is a documentation defect, not a loss: the
  first retry is at five seconds rather than the documented two.
- The browser half was run against the live agent on booklimo. It commits, and
  it sets back. If the last check is ever red, the field on booklimo.at is left
  holding the value the run typed and a person has to put it right.
- What is not covered here: two editors writing the *same* field at the same
  moment, which is the agent's rebase and belongs to `draft-state.md`; and a
  browser killed between the dispatch and the write, which cannot be staged
  from inside the page and is argued rather than measured.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [11]:
k.summary()
k.save_results('results-10-draft-persistence.json')
rc = k.verdict()
# Changes 2 and 3 of editing-performance.md have not been made yet, so this
# baseline run is expected to end here. The assertion stays: after the change
# every check in this notebook has to be green.
assert rc == 0, 'run has FAILures — see the summary above'

#,Status,Section,Check,Evidence
1,PASS,the harness,the harness bundles out of the jaen source,bundled 1723004 B
2,PASS,the persisted payload,the payload carries the unsent change and the edit,"the edit is in the payload, read back out of it. changes waiting in the outbox. what is waiting"
3,FAIL,the persisted payload,the payload does not carry the media catalogue,"77390 B payload, 140 media nodes in it, 76168 B of catalogue. expected 0, observed 140. change 2 of the plan: the catalogue is not persisted"
4,PASS,a reload,an edit survives the browser being started again,the value the second process read out of storage. and 140 media nodes came back with it
5,PASS,a hidden tab,a hidden tab has the edit in storage with no waiting,1 listener(s) heard visibilitychange. read back out of storage in the same frame as the event
6,PASS,a hidden tab,the edit is in storage within the idle deadline anyway,read back 300 ms after the dispatch. the write is synchronous today: it was there at once
7,PASS,offline,"an edit made offline is kept, in the store and in storage",read back out of storage while the agent is unreachable. waiting in the outbox. what the toolbar says
8,PASS,offline,the queue drains on its own when the agent comes back,the outbox is empty again. and the value is still the one that was typed. drained 5.0 s after the network came back
9,WARN,offline,the documented first retry is the one the code takes,"first retry at 5.0 s, while draft-state.md and the plan both say 2 s: RETRY_SECONDS is indexe..."
10,PASS,offline,the browser saying online drains it at once,the outbox is empty. the value read back out of storage. drained 25 ms after the online event


AssertionError: run has FAILures — see the summary above